In [57]:
import os
import sys

import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.express as px

current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
from src.models.filter_data.filter_data import *
from src.models.filter_data.feature_adder import *

In [72]:
csv_read_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")

df = pd.read_csv(csv_read_path)

In [ ]:
df_sum = df.groupby('datetime', as_index=False)['generation'].sum()
fig = px.scatter(df, x="datetime", y='forecasted_load',
                 title='forecasted_load over Time',
                 labels={'forecasted_load': 'Forecasted_load', 'datetime': 'Time'},
                 hover_data=['datetime', 'forecasted_load'])
fig.write_html(f"{project_root}/src/visualization/unit_figs/pre_process/forecasted_load.html")
#fig.show()

In [ ]:
df_sum = df.groupby('datetime', as_index=False)['generation'].sum()
fig = px.scatter(df_sum, x="datetime", y='generation',
                 title='Sum of generations over Time',
                 labels={'generation': 'Generation', 'datetime': 'Time'},
                 hover_data=['datetime', 'generation'])
#fig.write_html(f"{project_root}/src/visualization/unit_figs/pre_process/gen_sum.html")
fig.show()

In [75]:
df_count = df.groupby('datetime', as_index=False)['generation'].count()
fig = px.scatter(df_count, x="datetime", y='generation',
                 title='Count of generations over Time',
                 labels={'generation': 'Generation', 'datetime': 'Time'},
                 hover_data=['datetime', 'generation'])

fig.show()
#fig.write_html(f"{project_root}/src/visualization/unit_figs/pre_process/gen_count.html")

In [76]:
df['month'] = pd.to_datetime(df['date']).dt.month
df['year'] = pd.to_datetime(df['date']).dt.year
for i in range(1,13):
    m = i
    y = 2024
    df_selected = df[(df['month'] == m)&(df['year'] == y)]
    df_average_sum_in_month = df_selected.groupby('hour', as_index=False)['generation'].sum()
    fig = px.scatter(df_average_sum_in_month, x="hour", y='generation',
                    title=f'Sum of generations over Time {y}-{m}',
                    labels={'generation': 'Generation', 'hour': 'Hour'},
                    hover_data=['hour', 'generation'])

    #fig.write_html(f"{project_root}/src/visualization/unit_figs/pre_process/gen_sum.html")
    fig.show()

In [77]:
import plotly.graph_objects as go

fig = go.Figure()

df['month'] = pd.to_datetime(df['date']).dt.month
df['year'] = pd.to_datetime(df['date']).dt.year
for i in range(1,13):
    m = i
    y = 2024
    df_selected = df[(df['month'] == m)&(df['year'] == y)]
    df_average_sum_in_month = df_selected.groupby('hour', as_index=False)['generation'].sum()
    g20 = df_average_sum_in_month['generation'].iloc[19]
    trace = go.Scatter(
            x=df_average_sum_in_month['hour'],
            y=df_average_sum_in_month['generation'],
            mode='lines+markers',
            name=f'Month {i}',
            line=dict(width=2),
            marker=dict(size=6),
            showlegend=True
        )
        
    fig.add_trace(trace)

#fig.write_html(f"{project_root}/src/visualization/unit_figs/pre_process/gen_sum.html")
fig.show()

In [78]:
import pandas as pd
import numpy as np

def calculate_hourly_statistic(df, hour, statistic='mean'):
    """
    محاسبه آماره‌های مختلف برای یک ساعت خاص از روز، فارغ از تاریخ
    
    Parameters:
    -----------
    df : DataFrame
        دیتافریم حاوی داده‌ها
    hour : int
        ساعت مورد نظر (0-23)
    statistic : str
        نوع آماره مورد نظر:
        - 'mean': میانگین
        - 'median': میانه
        - 'std': انحراف معیار
        - 'var': واریانس
        - 'sum': مجموع
        - 'min': حداقل
        - 'max': حداکثر
        - 'count': تعداد
        
    Returns:
    --------
    float : مقدار آماره محاسبه شده
    """
    
    # اطمینان از تبدیل ستون date به datetime
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])
    
    # استخراج ساعت از تاریخ
    if 'hour' not in df.columns:
        df = df.copy()
        df['hour'] = df['date'].dt.hour
    
    # فیلتر کردن داده‌های ساعت مورد نظر
    df_hour = df[df['hour'] == hour]
    
    if df_hour.empty:
        return np.nan
    
    # محاسبه آماره مورد نظر
    statistic_functions = {
        'mean': lambda x: x['generation'].mean(),
        'median': lambda x: x['generation'].median(),
        'std': lambda x: x['generation'].std(),
        'var': lambda x: x['generation'].var(),
        'sum': lambda x: x['generation'].sum(),
        'min': lambda x: x['generation'].min(),
        'max': lambda x: x['generation'].max(),
        'count': lambda x: x['generation'].count()
    }
    
    if statistic in statistic_functions:
        return statistic_functions[statistic](df_hour)
    else:
        raise ValueError(f"آماره '{statistic}' پشتیبانی نمی‌شود. گزینه‌های معتبر: {list(statistic_functions.keys())}")


# تابع پیشرفته‌تر برای تحلیل همه ساعات
def analyze_all_hours(df, statistic='mean', include_hour_column=True):
    """
    محاسبه آماره برای تمام ساعات روز
    
    Parameters:
    -----------
    df : DataFrame
        دیتافریم حاوی داده‌ها
    statistic : str
        نوع آماره مورد نظر
    include_hour_column : bool
        آیا ستون hour به خروجی اضافه شود؟
        
    Returns:
    --------
    DataFrame : نتایج برای تمام ساعات
    """
    
    # اطمینان از وجود ستون hour
    if 'hour' not in df.columns:
        df = df.copy()
        df['hour'] = pd.to_datetime(df['date']).dt.hour
    
    # گروه‌بندی بر اساس ساعت و محاسبه آماره
    if statistic == 'mean':
        result = df.groupby('hour')['generation'].mean().reset_index()
    elif statistic == 'median':
        result = df.groupby('hour')['generation'].median().reset_index()
    elif statistic == 'std':
        result = df.groupby('hour')['generation'].std().reset_index()
    elif statistic == 'var':
        result = df.groupby('hour')['generation'].var().reset_index()
    elif statistic == 'sum':
        result = df.groupby('hour')['generation'].sum().reset_index()
    elif statistic == 'min':
        result = df.groupby('hour')['generation'].min().reset_index()
    elif statistic == 'max':
        result = df.groupby('hour')['generation'].max().reset_index()
    elif statistic == 'count':
        result = df.groupby('hour')['generation'].count().reset_index()
    else:
        raise ValueError(f"آماره '{statistic}' پشتیبانی نمی‌شود.")
    
    # تغییر نام ستون به آماره محاسبه شده
    result = result.rename(columns={'generation': statistic})
    
    if not include_hour_column:
        result = result.drop('hour', axis=1)
    
    return result


# تابع برای ایجاد نمودار
def plot_hourly_statistic(df, statistic='mean', title=None):
    """
    رسم نمودار آماره برای ساعات مختلف روز
    """
    import plotly.express as px
    
    # محاسبه آماره برای تمام ساعات
    df_stats = analyze_all_hours(df, statistic=statistic)
    
    # ایجاد عنوان پیش‌فرض
    if title is None:
        titles = {
            'mean': 'میانگین تولید در ساعات مختلف روز',
            'median': 'میانه تولید در ساعات مختلف روز',
            'std': 'انحراف معیار تولید در ساعات مختلف روز',
            'var': 'واریانس تولید در ساعات مختلف روز',
            'sum': 'مجموع تولید در ساعات مختلف روز',
            'min': 'حداقل تولید در ساعات مختلف روز',
            'max': 'حداکثر تولید در ساعات مختلف روز',
            'count': 'تعداد داده‌ها در ساعات مختلف روز'
        }
        title = titles.get(statistic, f'{statistic} تولید در ساعات مختلف روز')
    
    # ایجاد نمودار
    fig = px.bar(df_stats, x='hour', y=statistic,
                 title=title,
                 labels={'hour': 'ساعت', statistic: statistic.capitalize()},
                 hover_data=['hour', statistic])
    
    fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))
    
    return fig


# مثال استفاده:
if __name__ == "__main__":
    # فرض می‌کنیم df دیتافریم شماست
    # 1. محاسبه میانگین برای ساعت 14
    mean_14 = calculate_hourly_statistic(df, hour=14, statistic='mean')
    print(f"میانگین تولید در ساعت 14: {mean_14}")
    
    # 2. محاسبه واریانس برای ساعت 8
    var_8 = calculate_hourly_statistic(df, hour=8, statistic='var')
    print(f"واریانس تولید در ساعت 8: {var_8}")
    
    # 3. تحلیل کامل تمام ساعات
    hourly_means = analyze_all_hours(df, statistic='mean')
    print("میانگین تولید برای تمام ساعات:")
    print(hourly_means)
    
    # 4. رسم نمودار
    fig = plot_hourly_statistic(df, statistic='mean')
    fig.show()

میانگین تولید در ساعت 14: 97.38166615216129
واریانس تولید در ساعت 8: 2119.496241815545
میانگین تولید برای تمام ساعات:
    hour        mean
0      1   92.618233
1      2   88.926771
2      3   85.641893
3      4   83.219001
4      5   81.464696
5      6   79.398287
6      7   77.554070
7      8   81.911400
8      9   88.830670
9     10   93.708990
10    11   96.627389
11    12   98.075348
12    13   98.054546
13    14   97.381666
14    15   97.229097
15    16   96.820126
16    17   96.771006
17    18   97.879724
18    19   98.477353
19    20   98.534013
20    21  100.060671
21    22  100.233924
22    23   98.418118
23    24   95.679451
